In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def crawl_website(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted tags
    for tag in soup(["header", "footer", "nav", "script", "style", "aside"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    return text


In [2]:
!pip install --upgrade langchain langchain-community langchain-text-splitters


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                   ^^^^^^^^
  File "/usr/local/lib/py

In [12]:
!pip install -U langchain langchain-community langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [15]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_text(text)
    return chunks

In [17]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.6 MB/s eta 0:00:00


In [18]:
from sentence_transformers import SentenceTransformer
import faiss
import os
import pickle

model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(chunks):
    vectors = model.encode(chunks)
    return vectors

def save_embeddings(vectors, chunks):
    os.makedirs("data", exist_ok=True)

    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)

    faiss.write_index(index, "data/index.faiss")

    with open("data/chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)

def load_embeddings():
    index = faiss.read_index("data/index.faiss")
    with open("data/chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    return index, chunks

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
import pickle

def load_embeddings(file_path="embeddings.pkl"):
    with open(file_path, "rb") as f:
        return pickle.load(f)


In [25]:
%%writefile embeddings.py
import pickle

def load_embeddings(file_path="embeddings.pkl"):
    with open(file_path, "rb") as f:
        return pickle.load(f)


Writing embeddings.py


In [27]:
from embeddings import load_embeddings
print("Embeddings module loaded successfully!")


Embeddings module loaded successfully!


In [28]:
import sys
sys.path.append('/content')


In [29]:
from embeddings import load_embeddings


In [30]:
from sentence_transformers import SentenceTransformer
import numpy as np
from embeddings import load_embeddings

model = SentenceTransformer("all-MiniLM-L6-v2")

def answer_question(question):
    try:
        index, chunks = load_embeddings()
    except:
        return "Please index a website first."

    q_vector = model.encode([question])
    D, I = index.search(q_vector, k=3)

    context = " ".join([chunks[i] for i in I[0]])

    if len(context.strip()) == 0:
        return "The answer is not available on the provided website."

    return context[:500]  # simple answer (LLM can be added later)


In [32]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.9 MB/s eta 0:00:00


In [33]:
!pip install streamlit pyngrok


In [34]:
%%writefile app.py
import streamlit as st

st.title("Hello Streamlit in Colab!")
st.write("Streamlit is working!")


Writing app.py


In [38]:
from pyngrok import ngrok
ngrok.set_auth_token("2yShxu6XhJhYAiXxl1XO6VVPKgL_72tfC1wDAqGhDDWj7J8PY")


In [41]:
%%writefile crawler.py
import requests
from bs4 import BeautifulSoup

def crawl_website(url):
    """
    Takes a website URL and returns all text content from the page.
    """
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup.get_text()


Writing crawler.py


In [43]:
%%writefile processor.py
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_text(text):
    """
    Splits text into chunks for embeddings or search.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    return splitter.split_text(text)


Writing processor.py


In [45]:
from processor import process_text

text = "This is a test text to split into chunks."
chunks = process_text(text)
print(chunks)


['This is a test text to split into chunks.']


In [56]:
%%writefile embeddings.py
from sentence_transformers import SentenceTransformer
import pickle

model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(chunks):
    return model.encode(chunks)

def save_embeddings(embeddings, file_path="embeddings.pkl"):
    with open(file_path, "wb") as f:
        pickle.dump(embeddings, f)

def load_embeddings(file_path="embeddings.pkl"):
    with open(file_path, "rb") as f:
        return pickle.load(f)


Overwriting embeddings.py


In [57]:
# Remove cached module
import sys
if 'embeddings' in sys.modules:
    del sys.modules['embeddings']


In [58]:
from embeddings import create_embeddings, save_embeddings, load_embeddings

# Test
chunks = ["Hello world", "This is a test"]
embs = create_embeddings(chunks)
print("Embeddings shape:", embs.shape)


Embeddings shape: (2, 384)


In [60]:
%%writefile chatbot.py
import numpy as np

def answer_question(question, chunks, embeddings, model):
    """
    Simple RAG-style retrieval using cosine similarity.
    Returns the chunk most similar to the question.
    """
    # Get embedding for the question
    q_emb = model.encode([question])

    # Compute cosine similarity with existing embeddings
    scores = np.dot(embeddings, q_emb.T).flatten()

    # Get the index of the best matching chunk
    best_idx = np.argmax(scores)

    return chunks[best_idx]


Writing chatbot.py


In [61]:
from chatbot import answer_question

print("Chatbot module imported successfully!")


Chatbot module imported successfully!


In [62]:
import streamlit as st
from crawler import crawl_website
from processor import process_text
from embeddings import create_embeddings, save_embeddings
from chatbot import answer_question

st.title("🌐 Website AI Chatbot")


2026-01-30 08:32:31.257 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:31.609 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-01-30 08:32:31.610 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:31.612 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [63]:
import streamlit as st
from crawler import crawl_website
from processor import process_text
from embeddings import create_embeddings, save_embeddings
from chatbot import answer_question

st.title("🌐 Website AI Chatbot")

url = st.text_input("Enter Website URL:")

if st.button("Index Website"):
    text = crawl_website(url)

    if text is None:
        st.error("Invalid or unreachable URL.")
    else:
        chunks = process_text(text)
        vectors = create_embeddings(chunks)
        save_embeddings(vectors, chunks)
        st.success("Website Indexed Successfully!")

question = st.text_input("Ask a question:")

if st.button("Ask"):
    answer = answer_question(question)
    st.write("🤖 Answer:", answer)


2026-01-30 08:32:36.459 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-30 08:32:36.474 Session state does not function when running a script without `streamlit run`
2026-01-30 08:32